# Deadline Regression Discontinuity Analysis

**Research Question**: Does missing the estimated delivery deadline *cause* customers to leave lower review scores?

---

## The Problem: Correlation ≠ Causation

If we simply compare late vs on-time deliveries, we'll find that late orders have **much worse reviews** (~1.7 stars lower). But can we conclude that being late *caused* the bad reviews?

**No!** Because late orders might be systematically different:

| Confounder | Causes Lateness | Also Causes Bad Reviews |
|------------|-----------------|------------------------|
| Remote destination | Longer shipping routes | Different customer expectations |
| Heavy/fragile items | Logistics complications | More likely damaged |
| Bad sellers | Slow processing | Poor product quality |
| Peak seasons | Overwhelmed logistics | Stock/quality issues |

The raw comparison mixes the **causal effect of lateness** with all these **confounding factors**.

---

## The Solution: Regression Discontinuity (RD)

### Key Insight

Orders that arrive **just barely late** (e.g., 0.5 days after deadline) vs **just barely on-time** (e.g., 0.5 days before deadline) are almost identical in every other way. The only reason one landed slightly late and the other slightly early is **random noise** — traffic, weather, a sorting facility running slow.

This is like a **natural experiment**: near the cutoff, "treatment" (being late) is essentially randomly assigned.

### Analogy: Scholarship Test Score

Imagine a scholarship requires a test score ≥ 70:
- Student A scores 69 → No scholarship
- Student B scores 71 → Gets scholarship

These students are nearly identical in ability. The 2-point difference is likely due to luck (slept well, lucky guesses). Comparing their future outcomes gives us the **causal effect of the scholarship**.

But comparing someone who scored 40 vs 95? They're fundamentally different people — the comparison tells us nothing about the scholarship.

---

## RD Setup for This Analysis

| Component | Value |
|-----------|-------|
| **Running Variable** | `delivery_delay_days` (actual − estimated delivery) |
| **Cutoff** | 0 days |
| **Treatment** | Being labeled "late" (delay > 0) |
| **Outcome** | Review score (1-5 stars) |

**Key Assumption**: Near the cutoff, being late vs on-time is determined by random factors, not systematic differences.

---
## 1. Setup and Data Loading

In [ ]:
# Import libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Plotly for interactive visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_all_tables, create_analysis_dataset
from src.analysis.rd import (
    estimate_rd_effect,
    mccrary_density_test,
    rd_sensitivity_analysis,
    optimal_bandwidth,
    triangular_kernel,
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Color scheme
COLORS = {
    'on_time': '#2ecc71',   # Green for on-time
    'late': '#e74c3c',      # Red for late
    'neutral': '#3498db',   # Blue
    'cutoff': '#2c3e50',    # Dark for cutoff line
}

print("Libraries loaded successfully!")

In [ ]:
# Load data
print("Loading data...")
tables = load_all_tables()
df = create_analysis_dataset(tables)
print(f"Total orders loaded: {len(df):,}")

---
## 2. Prepare RD Dataset

### Why are we doing this?

We need to:
1. **Filter to delivered orders** — only delivered orders have actual delivery dates
2. **Require delay data** — we need to know how many days from the deadline
3. **Require review data** — reviews are our outcome variable

This gives us orders where we can measure both the "treatment" (late/on-time) and the "outcome" (review score).

In [ ]:
# Filter to delivered orders with delay and review information
rd_data = df[
    (df['order_status'] == 'delivered') &
    (df['days_from_deadline'].notna()) &
    (df['review_score'].notna())
].copy()

# Create clear variable names
rd_data['delivery_delay_days'] = rd_data['days_from_deadline']
rd_data['is_late'] = (rd_data['delivery_delay_days'] > 0).astype(int)

print(f"Delivered orders with delay & review data: {len(rd_data):,}")
print(f"\nDelay range: [{rd_data['delivery_delay_days'].min():.1f}, {rd_data['delivery_delay_days'].max():.1f}] days")
print(f"Late deliveries: {rd_data['is_late'].sum():,} ({rd_data['is_late'].mean()*100:.1f}%)")
print(f"On-time deliveries: {(rd_data['is_late'] == 0).sum():,} ({(1-rd_data['is_late'].mean())*100:.1f}%)")

In [ ]:
# Summary statistics by side of cutoff
summary = rd_data.groupby('is_late').agg({
    'review_score': ['count', 'mean', 'std', 'median'],
    'delivery_delay_days': ['mean', 'min', 'max'],
    'total_value': 'mean',
    'total_freight': 'mean',
}).round(3)

summary.index = ['On-Time (≤0 days)', 'Late (>0 days)']
print("Summary Statistics by Delivery Status:")
summary

---
## 3. The Raw Comparison (Why It's Biased)

### Why are we doing this?

Before doing RD, let's see what a **naive comparison** would tell us. This is what most people would do: simply compare average reviews for late vs on-time orders.

This comparison is **biased** because it includes:
- Orders that are 1 day late (similar to on-time)
- Orders that are 50 days late (very different from on-time!)

Orders that are very late likely have **other problems** (bad sellers, remote locations, etc.) that ALSO cause bad reviews.

In [ ]:
# Raw difference in review scores
on_time = rd_data[rd_data['is_late'] == 0]
late = rd_data[rd_data['is_late'] == 1]

raw_diff = late['review_score'].mean() - on_time['review_score'].mean()
raw_se = np.sqrt(late['review_score'].var()/len(late) + on_time['review_score'].var()/len(on_time))
raw_tstat = raw_diff / raw_se
raw_pvalue = 2 * (1 - stats.norm.cdf(abs(raw_tstat)))

print("=" * 50)
print("RAW COMPARISON (potentially biased!)")
print("=" * 50)
print(f"Mean review (on-time): {on_time['review_score'].mean():.3f} stars")
print(f"Mean review (late):    {late['review_score'].mean():.3f} stars")
print(f"\nRaw difference:        {raw_diff:.3f} stars")
print(f"t-statistic:           {raw_tstat:.2f}")
print(f"p-value:               {raw_pvalue:.4f}")
print("\n⚠️  This comparison is BIASED - late orders differ systematically!")

### Let's PROVE the bias exists

If late orders were **randomly** late (just bad luck), their other characteristics (order value, freight, etc.) should look similar to on-time orders.

But if late orders are **systematically different**, we'll see that very late orders have different characteristics — proving the raw comparison is confounded.

In [ ]:
# Compare characteristics across delay groups to show systematic differences
rd_data['delay_group'] = pd.cut(
    rd_data['delivery_delay_days'],
    bins=[-200, -10, 0, 5, 15, 200],
    labels=['Very Early (<-10d)', 'On-Time (-10 to 0d)', 'Slightly Late (0-5d)', 'Moderately Late (5-15d)', 'Very Late (>15d)']
)

comparison = rd_data.groupby('delay_group', observed=True).agg({
    'review_score': 'mean',
    'total_value': 'mean',
    'total_freight': 'mean',
    'n_items': 'mean',
    'delivery_time_actual': 'mean',
}).round(2)

comparison.columns = ['Avg Review', 'Avg Order Value (R$)', 'Avg Freight (R$)', 'Avg Items', 'Avg Delivery Days']
print("Characteristics by Delay Group:")
print("(If late orders were randomly late, these columns would be similar across rows)")
print()
comparison

### What does this table tell us?

Notice how **Very Late** orders have:
- Higher freight costs → likely going to harder-to-reach places
- Longer delivery times → systematically more complex deliveries

These orders aren't just "unlucky" — they're fundamentally different. That's why the raw comparison is biased.

**The RD solution**: Only compare orders that are *nearly identical* (just barely late vs just barely on-time).

---
## 4. Visualize the Running Variable

### Why are we doing this?

Before running RD, we need to understand the distribution of our **running variable** (delivery delay). This helps us:
1. See where most data is concentrated
2. Identify potential issues (outliers, gaps)
3. Visualize the relationship between delay and reviews

In [ ]:
# Distribution of delivery delay
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=on_time['delivery_delay_days'],
    name='On-Time',
    marker_color=COLORS['on_time'],
    opacity=0.7,
    nbinsx=100,
))

fig.add_trace(go.Histogram(
    x=late['delivery_delay_days'],
    name='Late',
    marker_color=COLORS['late'],
    opacity=0.7,
    nbinsx=100,
))

fig.add_vline(x=0, line_dash="dash", line_color=COLORS['cutoff'], line_width=2,
              annotation_text="Deadline (cutoff)", annotation_position="top")

fig.update_layout(
    title='Distribution of Delivery Delay (Running Variable)',
    xaxis_title='Delivery Delay (days, negative = early)',
    yaxis_title='Number of Orders',
    barmode='overlay',
    xaxis=dict(range=[-60, 60]),
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)

fig.show()

In [ ]:
# Review score vs delay (binned scatter)
# This is the KEY visualization for RD - we're looking for a JUMP at the cutoff

rd_data['delay_bin'] = pd.cut(rd_data['delivery_delay_days'], bins=100)
binned = rd_data.groupby('delay_bin', observed=True).agg({
    'review_score': ['mean', 'std', 'count'],
    'delivery_delay_days': 'mean',
}).reset_index()
binned.columns = ['bin', 'mean_review', 'std_review', 'count', 'mean_delay']
binned = binned.dropna()

binned_left = binned[binned['mean_delay'] <= 0]
binned_right = binned[binned['mean_delay'] > 0]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=binned_left['mean_delay'], y=binned_left['mean_review'],
    mode='markers', marker=dict(color=COLORS['on_time'], size=8, opacity=0.7),
    name='On-Time',
))

fig.add_trace(go.Scatter(
    x=binned_right['mean_delay'], y=binned_right['mean_review'],
    mode='markers', marker=dict(color=COLORS['late'], size=8, opacity=0.7),
    name='Late',
))

fig.add_vline(x=0, line_dash="dash", line_color=COLORS['cutoff'], line_width=2)

fig.update_layout(
    title='Average Review Score by Delivery Delay<br><sup>Look for a JUMP at the cutoff (x=0) — that would be the causal effect</sup>',
    xaxis_title='Delivery Delay (days)',
    yaxis_title='Average Review Score',
    xaxis=dict(range=[-40, 40]),
    yaxis=dict(range=[1, 5]),
)

fig.show()

### What to look for in this plot

- **If there's a causal effect**: We'd see a sharp **jump** (discontinuity) at x=0
- **If there's no causal effect**: The line would be smooth across the cutoff

Notice how reviews decline as delay increases — but is there a **sudden jump** exactly at the deadline? That's what RD will measure.

---
## 5. McCrary Density Test (Can People Manipulate the Cutoff?)

### Why are we doing this?

RD only works if people **cannot precisely manipulate** whether they land just above or below the cutoff.

**Example of manipulation**: If sellers could choose exactly when orders arrive, they'd bunch deliveries *just* before the deadline to avoid the "late" label.

The McCrary test checks: **Is there a suspicious jump in the number of observations at the cutoff?**

```
If NO manipulation:          If MANIPULATION:
                             
Density                      Density
   |    ___                     |    ___
   |   /   \                    |   /   |___
   |  /     \                   |  /         \
   | /       \                  | /           \
   |/_________\                 |/______|______\
          0                            0
   (smooth)                    (jump = manipulation!)
```

In [ ]:
# McCrary density test
mccrary = mccrary_density_test(
    rd_data['delivery_delay_days'].values,
    cutoff=0,
    bandwidth=5,
    n_bins=100,
)

print("=" * 50)
print("McCRARY DENSITY TEST")
print("=" * 50)
print(f"H0: No discontinuity in density at cutoff (no manipulation)")
print(f"\nDiscontinuity estimate: {mccrary['discontinuity']:.4f}")
print(f"t-statistic:            {mccrary['t_statistic']:.2f}")
print(f"p-value:                {mccrary['pvalue']:.4f}")
print(f"\nInterpretation: {mccrary['interpretation']}")

In [ ]:
# Visualize the density test
fig = go.Figure()

colors = [COLORS['on_time'] if x < 0 else COLORS['late'] for x in mccrary['bin_centers']]
fig.add_trace(go.Bar(
    x=mccrary['bin_centers'],
    y=mccrary['density'],
    marker_color=colors,
    name='Density',
))

fig.add_vline(x=0, line_dash="dash", line_color=COLORS['cutoff'], line_width=2)

fig.update_layout(
    title=f'McCrary Density Test: Is There Bunching at the Cutoff?<br><sup>p={mccrary["pvalue"]:.4f} — {mccrary["interpretation"]}</sup>',
    xaxis_title='Delivery Delay (days)',
    yaxis_title='Density',
    showlegend=False,
)

fig.show()

### What does this mean?

- **If p > 0.05**: No evidence of manipulation → RD is valid ✓
- **If p < 0.05**: Evidence of manipulation → RD results should be interpreted cautiously ⚠️

Possible reasons for density discontinuity (even without intentional manipulation):
1. Sellers naturally try to meet deadlines
2. Delivery scheduling patterns (daily deliveries)
3. The deadline itself affects logistics decisions

---
## 6. Choosing the Bandwidth

### Why are we doing this?

The **bandwidth** is the most important choice in RD. It determines how close to the cutoff we look.

| Bandwidth | Pros | Cons |
|-----------|------|------|
| **Too small** | Very comparable observations | Few observations → noisy estimate |
| **Too large** | Many observations → precise | Includes dissimilar observations → biased |

We use the **Imbens-Kalyanaraman (IK)** method to automatically find the optimal bandwidth that balances this trade-off.

### The formula balances:
- **Curvature**: If the relationship is curved, we need smaller bandwidth
- **Variance**: If data is noisy, we can use larger bandwidth
- **Sample size**: More data allows smaller bandwidth

In [ ]:
# Calculate optimal bandwidth
h_opt = optimal_bandwidth(
    rd_data['delivery_delay_days'].values,
    rd_data['review_score'].values,
    cutoff=0,
    method='ik',
)

print(f"Optimal bandwidth (IK): {h_opt:.3f} days")
print(f"\nThis means we compare orders with delay in:")
print(f"  On-time window: [{-h_opt:.2f}, 0] days")
print(f"  Late window:    [0, +{h_opt:.2f}] days")
print(f"\n'Just barely on-time' vs 'Just barely late'")

In [ ]:
# How many observations are we actually using?
n_left = ((rd_data['delivery_delay_days'] <= 0) & (rd_data['delivery_delay_days'] >= -h_opt)).sum()
n_right = ((rd_data['delivery_delay_days'] > 0) & (rd_data['delivery_delay_days'] <= h_opt)).sum()

print(f"Observations within optimal bandwidth:")
print(f"  Left (just on-time):   {n_left:,}")
print(f"  Right (just late):     {n_right:,}")
print(f"  Total for RD estimate: {n_left + n_right:,}")
print(f"\nThis is {(n_left + n_right)/len(rd_data)*100:.1f}% of our data")
print(f"\nWe're throwing away {100 - (n_left + n_right)/len(rd_data)*100:.1f}% of data to get an UNBIASED estimate!")

In [ ]:
# Visualize what the bandwidth looks like
fig = go.Figure()

# All data (grayed out)
fig.add_trace(go.Scatter(
    x=binned['mean_delay'], y=binned['mean_review'],
    mode='markers', marker=dict(color='lightgray', size=6),
    name='Outside bandwidth (not used)',
))

# Data within bandwidth (highlighted)
bw_mask = abs(binned['mean_delay']) <= h_opt
fig.add_trace(go.Scatter(
    x=binned.loc[bw_mask & (binned['mean_delay'] <= 0), 'mean_delay'],
    y=binned.loc[bw_mask & (binned['mean_delay'] <= 0), 'mean_review'],
    mode='markers', marker=dict(color=COLORS['on_time'], size=12),
    name='Just on-time (USED)',
))

fig.add_trace(go.Scatter(
    x=binned.loc[bw_mask & (binned['mean_delay'] > 0), 'mean_delay'],
    y=binned.loc[bw_mask & (binned['mean_delay'] > 0), 'mean_review'],
    mode='markers', marker=dict(color=COLORS['late'], size=12),
    name='Just late (USED)',
))

# Mark the bandwidth window
fig.add_vrect(x0=-h_opt, x1=h_opt, fillcolor="yellow", opacity=0.15, line_width=0)
fig.add_vline(x=0, line_dash="dash", line_color=COLORS['cutoff'], line_width=2)
fig.add_vline(x=-h_opt, line_dash="dot", line_color='orange', line_width=1)
fig.add_vline(x=h_opt, line_dash="dot", line_color='orange', line_width=1)

fig.update_layout(
    title=f'The Bandwidth Window: Only Yellow Region Is Used<br><sup>Bandwidth = ±{h_opt:.2f} days from cutoff</sup>',
    xaxis_title='Delivery Delay (days)',
    yaxis_title='Average Review Score',
    xaxis=dict(range=[-15, 15]),
    yaxis=dict(range=[2.5, 5]),
)

fig.show()

### Key insight from this visualization

- **Gray points**: Thrown away — too far from cutoff to be comparable
- **Colored points in yellow region**: Used for estimation — these are comparable

We're essentially asking: "Among orders that were almost exactly on the deadline, did the slightly late ones get worse reviews?"

---
## 7. Main RD Estimation

### Why are we doing this?

Now we estimate the actual causal effect. The method:

1. **Take observations within bandwidth** (just late vs just on-time)
2. **Weight by distance**: Observations closer to cutoff get more weight (triangular kernel)
3. **Fit separate lines** on each side of the cutoff
4. **Measure the gap** between the lines at the cutoff = **causal effect (τ)**

```
Review
  5 |          
    |    ___________
    |   /           
  4 |  / ← on-time line
    |       ↓ τ (the gap we measure)
  3 |       _________
    |      /         
    |     / ← late line
  2 |____/___________
      -h    0    +h   delay
```

In [ ]:
# Main RD estimation
rd_result = estimate_rd_effect(
    df=rd_data,
    running_var='delivery_delay_days',
    outcome='review_score',
    cutoff=0,
    bandwidth=h_opt,
    polynomial_order=1,  # Local linear regression
    kernel='triangular',  # More weight to observations near cutoff
)

print("=" * 60)
print("MAIN RD RESULTS")
print("=" * 60)
print(f"\nMethod: Local linear regression, triangular kernel")
print(f"Bandwidth: {rd_result.bandwidth:.3f} days")
print(f"\n{'-' * 40}")
print(f"RD Estimate (τ):    {rd_result.estimate:.4f} stars")
print(f"Standard Error:     {rd_result.se:.4f}")
print(f"95% CI:             [{rd_result.ci_low:.4f}, {rd_result.ci_high:.4f}]")
print(f"p-value:            {rd_result.pvalue:.4f}")
print(f"{'-' * 40}")
print(f"\nObservations used:")
print(f"  Just on-time:     {rd_result.n_left:,}")
print(f"  Just late:        {rd_result.n_right:,}")
print(f"  Total:            {rd_result.n_effective:,}")

### How to interpret the results

| Metric | Meaning |
|--------|--------|
| **RD Estimate (τ)** | The causal effect of being late at the cutoff |
| **Standard Error** | Uncertainty in our estimate |
| **95% CI** | True effect is likely within this range |
| **p-value** | Probability of seeing this result if true effect = 0 |

In [ ]:
# Interpretation
alpha = 0.05
is_significant = rd_result.pvalue < alpha

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

if is_significant:
    direction = "decrease" if rd_result.estimate < 0 else "increase"
    print(f"\n✓ STATISTICALLY SIGNIFICANT at α = {alpha}")
    print(f"\nConclusion: Crossing the deadline CAUSES a {abs(rd_result.estimate):.3f} star")
    print(f"{direction} in review scores.")
else:
    print(f"\n✗ NOT STATISTICALLY SIGNIFICANT at α = {alpha}")
    print(f"\nConclusion: We find NO evidence that barely missing the deadline")
    print(f"causes lower reviews. The effect at the cutoff is approximately ZERO.")
    
print(f"\n" + "-" * 60)
print("COMPARISON: Raw vs RD Estimate")
print("-" * 60)
print(f"Raw difference (ALL late vs ALL on-time): {raw_diff:.3f} stars")
print(f"RD estimate (JUST late vs JUST on-time): {rd_result.estimate:.3f} stars")
print(f"\nThe raw difference is {abs(raw_diff/rd_result.estimate) if rd_result.estimate != 0 else 'infinitely'}x larger!")
print(f"This shows how much BIAS was in the raw comparison.")

---
## 8. Visualize the RD Result

In [ ]:
# Create the main RD visualization
fig = go.Figure()

# Binned scatter
fig.add_trace(go.Scatter(
    x=binned_left['mean_delay'], y=binned_left['mean_review'],
    mode='markers', marker=dict(color=COLORS['on_time'], size=8, opacity=0.6),
    name='On-Time (binned)',
))

fig.add_trace(go.Scatter(
    x=binned_right['mean_delay'], y=binned_right['mean_review'],
    mode='markers', marker=dict(color=COLORS['late'], size=8, opacity=0.6),
    name='Late (binned)',
))

# Polynomial fits
left_data = rd_data[rd_data['delivery_delay_days'] <= 0]
right_data = rd_data[rd_data['delivery_delay_days'] > 0]

x_left = np.linspace(max(left_data['delivery_delay_days'].min(), -30), 0, 100)
coef_left = np.polyfit(left_data['delivery_delay_days'], left_data['review_score'], 2)
fig.add_trace(go.Scatter(
    x=x_left, y=np.polyval(coef_left, x_left),
    mode='lines', line=dict(color=COLORS['on_time'], width=3),
    name='Fit (on-time)',
))

x_right = np.linspace(0, min(right_data['delivery_delay_days'].max(), 30), 100)
coef_right = np.polyfit(right_data['delivery_delay_days'], right_data['review_score'], 2)
fig.add_trace(go.Scatter(
    x=x_right, y=np.polyval(coef_right, x_right),
    mode='lines', line=dict(color=COLORS['late'], width=3),
    name='Fit (late)',
))

fig.add_vline(x=0, line_dash="dash", line_color=COLORS['cutoff'], line_width=2)
fig.add_annotation(x=0, y=5, text="Deadline", showarrow=False, yshift=10)

sig = "*" if rd_result.pvalue < 0.05 else "(n.s.)"
fig.update_layout(
    title=f'RD Plot: Effect of Late Delivery on Reviews<br>'
          f'<sup>τ = {rd_result.estimate:.3f} {sig}, p = {rd_result.pvalue:.3f}</sup>',
    xaxis_title='Delivery Delay (days)',
    yaxis_title='Average Review Score',
    xaxis=dict(range=[-30, 30]),
    yaxis=dict(range=[1, 5]),
)

fig.show()

---
## 9. Sensitivity Analysis: Is Our Result Robust?

### Why are we doing this?

Our result depends on choices we made:
- Bandwidth (how close to cutoff)
- Polynomial order (linear vs quadratic fit)
- Kernel (triangular vs uniform weighting)

**If our result is real**, it should be similar across different reasonable choices.

**If our result is fragile**, it will change dramatically with small changes in methodology.

In [ ]:
# Run sensitivity analysis
sensitivity = rd_sensitivity_analysis(
    df=rd_data,
    running_var='delivery_delay_days',
    outcome='review_score',
    cutoff=0,
    bandwidth_range=[h_opt * m for m in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]],
    polynomial_orders=[1, 2],
    kernels=['triangular', 'uniform'],
)

print("Sensitivity Analysis: How does the estimate change with different specifications?")
print("=" * 80)
display_cols = ['bandwidth_ratio', 'polynomial_order', 'kernel', 'estimate', 'se', 'pvalue', 'significant']
sensitivity[display_cols].round(4)

In [ ]:
# Visualize sensitivity
fig = go.Figure()

for kernel in sensitivity['kernel'].unique():
    for poly in sensitivity['polynomial_order'].unique():
        subset = sensitivity[(sensitivity['kernel'] == kernel) & (sensitivity['polynomial_order'] == poly)]
        fig.add_trace(go.Scatter(
            x=subset['bandwidth_ratio'],
            y=subset['estimate'],
            mode='lines+markers',
            name=f'{kernel}, poly={poly}',
            error_y=dict(type='data', array=1.96 * subset['se'], visible=True),
        ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_vline(x=1.0, line_dash="dot", line_color="blue",
              annotation_text="Optimal BW", annotation_position="top")

fig.update_layout(
    title='Sensitivity: Does the Estimate Change with Different Specifications?<br>'
          '<sup>If lines stay near 0 across all bandwidths → robust null result</sup>',
    xaxis_title='Bandwidth (multiple of optimal)',
    yaxis_title='RD Estimate (τ)',
)

fig.show()

# Summary
n_sig = sensitivity['significant'].sum()
print(f"\nOut of {len(sensitivity)} specifications tested:")
print(f"  {n_sig} are statistically significant ({n_sig/len(sensitivity)*100:.0f}%)")
print(f"  {len(sensitivity)-n_sig} are NOT significant ({(len(sensitivity)-n_sig)/len(sensitivity)*100:.0f}%)")

---
## 10. Covariate Balance Check

### Why are we doing this?

If RD is working correctly, observations just above and just below the cutoff should be **similar in all pre-treatment characteristics**.

If they're NOT similar, it suggests the "random assignment" assumption is violated.

In [ ]:
# Check balance of covariates near the cutoff
near_cutoff = rd_data[abs(rd_data['delivery_delay_days']) <= h_opt].copy()

covariates = ['total_value', 'total_freight', 'n_items', 'total_price']
balance_results = []

for cov in covariates:
    if cov in near_cutoff.columns:
        left_vals = near_cutoff[near_cutoff['is_late'] == 0][cov].dropna()
        right_vals = near_cutoff[near_cutoff['is_late'] == 1][cov].dropna()
        
        if len(left_vals) > 0 and len(right_vals) > 0:
            t_stat, p_val = stats.ttest_ind(left_vals, right_vals)
            balance_results.append({
                'Covariate': cov,
                'Mean (Just On-Time)': left_vals.mean(),
                'Mean (Just Late)': right_vals.mean(),
                'Difference': right_vals.mean() - left_vals.mean(),
                'p-value': p_val,
                'Balanced?': '✓ Yes' if p_val > 0.05 else '✗ No',
            })

balance_df = pd.DataFrame(balance_results)
print("Covariate Balance Check (within optimal bandwidth):")
print("If RD is valid, these groups should be similar (p > 0.05)")
print("=" * 80)
balance_df.round(3)

---
## 11. Final Summary

### What did we learn?

In [ ]:
print("=" * 70)
print("DEADLINE RD ANALYSIS - FINAL SUMMARY")
print("=" * 70)

print("\n📊 DATA")
print(f"   Total delivered orders:     {len(rd_data):,}")
print(f"   Late deliveries:            {rd_data['is_late'].sum():,} ({rd_data['is_late'].mean()*100:.1f}%)")
print(f"   Used for RD estimate:       {rd_result.n_effective:,} ({rd_result.n_effective/len(rd_data)*100:.1f}%)")

print("\n📏 RAW COMPARISON (biased)")
print(f"   Mean review (on-time):      {on_time['review_score'].mean():.3f} stars")
print(f"   Mean review (late):         {late['review_score'].mean():.3f} stars")
print(f"   Raw difference:             {raw_diff:.3f} stars ***")

print("\n🎯 RD ESTIMATE (causal)")
print(f"   Bandwidth:                  ±{rd_result.bandwidth:.2f} days")
print(f"   RD estimate (τ):            {rd_result.estimate:.4f} stars")
print(f"   95% CI:                     [{rd_result.ci_low:.4f}, {rd_result.ci_high:.4f}]")
print(f"   p-value:                    {rd_result.pvalue:.4f}")
print(f"   Significant at 5%:          {'Yes' if rd_result.pvalue < 0.05 else 'No'}")

print("\n⚠️  VALIDITY CHECK")
print(f"   McCrary test p-value:       {mccrary['pvalue']:.4f}")
print(f"   Manipulation evidence:      {mccrary['interpretation']}")

print("\n" + "=" * 70)
print("KEY TAKEAWAY")
print("=" * 70)

if rd_result.pvalue >= 0.05:
    print("""
    The large raw difference (-1.73 stars) is MISLEADING.
    
    When we properly compare "just barely late" vs "just barely on-time" orders
    (which are comparable in all other respects), we find NO significant effect.
    
    This means:
    • Barely missing the deadline doesn't hurt reviews
    • The raw difference is driven by VERY late orders
    • VERY late orders have other problems (bad sellers, remote areas, etc.)
    
    Business implication: Focus on preventing VERY late deliveries (10+ days),
    not on barely meeting deadlines.
    """)
else:
    print(f"""
    Crossing the deadline DOES cause a {abs(rd_result.estimate):.3f} star penalty.
    
    This is a causal effect — even when comparing nearly identical orders,
    the "late" label itself causes worse reviews.
    """)

---
## References

1. **Imbens & Lemieux (2008)**. Regression discontinuity designs: A guide to practice. *Journal of Econometrics*.
2. **Lee & Lemieux (2010)**. Regression discontinuity designs in economics. *Journal of Economic Literature*.
3. **McCrary (2008)**. Manipulation of the running variable in RD. *Journal of Econometrics*.
4. **Cattaneo, Idrobo & Titiunik (2019)**. A Practical Introduction to RD Designs. *Cambridge University Press*.